# Hướng Dẫn Thực Nghiệm Colab Toàn Diện (Testing & Training)

> **Lưu ý:**
> 1. **Runtime GPU:** Chọn **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
> 2. **Lối tắt Drive:** Hãy tạo lối tắt (Shortcut) thư mục `KLTN-2026-testingNtraining` vào `My Drive` của bạn trước khi chạy.

## Bước 1: Kết nối Google Drive, Kiểm tra & Setup Môi trường

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SHARED_DRIVE = "/content/drive/MyDrive/KLTN-2026-testingNtraining"

if not os.path.exists(SHARED_DRIVE):
    raise FileNotFoundError(
        "❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!\n"
        "👉 Hãy vào Google Drive web -> 'Được chia sẻ với tôi' -> Chuột phải vào 'KLTN-2026-testingNtraining' "
        "-> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này."
    )

print(f"✅ Đã kết nối thành công với thư mục chung: {SHARED_DRIVE}")
os.makedirs(f"{SHARED_DRIVE}/runs/baseline", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/backbone", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/neck", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/head", exist_ok=True)
print("✅ Các thư mục gói (runs/baseline, runs/backbone, runs/neck, runs/head) đã sẵn sàng!\n")

# Clone repo & Cài đặt thư viện
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!pip install -q ultralytics albumentations tabulate

## Bước 2 (Chỉ chạy 1 LẦN DUY NHẤT bởi Lead): Ghép 5 Part Dữ Liệu Lên Drive Chung
> ⚠️ **LƯU Ý:**
> - Cell này chỉ cần **Trưởng nhóm (Lead) chạy đúng 1 lần** trên Google Drive dùng chung sau khi tải 5 part dữ liệu lên.
> - Khi file zip hợp nhất `mvtec_augmented.zip` đã có sẵn trong Drive chung, các thành viên khác **bỏ qua cell này**.

In [ ]:
# Ghép 5 part trên Drive thành file zip duy nhất lưu trực tiếp trên Drive dùng chung
!cat /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented_2.zip.00* > /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented.zip
!ls -lh /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented.zip
print("✅ Đã ghép xong 5 part thành 1 file zip duy nhất trên Drive chung cho cả nhóm!")

## Bước 3: Giải Nén Dữ Liệu Lên SSD Colab (Mọi Thành Viên Đều Chạy)
> Giải nén từ file zip trên Drive vào phân vùng SSD tốc độ cao của Colab để tối ưu tốc độ đọc ảnh khi training.

In [ ]:
!mkdir -p data/processed/split_70_15_15_augmented

# Ưu tiên giải nén từ file zip hợp nhất (nếu đã ghép), fallback sang ghép nhanh nếu chưa có
import os
zip_merged = "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented.zip"

if os.path.exists(zip_merged):
    print("📦 Phát hiện file zip hợp nhất. Đang giải nén lên SSD Colab...")
    !unzip -q "{zip_merged}" -d data/processed/split_70_15_15_augmented/
else:
    print("⚙️ Chưa có file ghép sẵn, đang ghép tạm và giải nén...")
    !cat /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented_2.zip.00* > /content/archive_temp.zip
    !7z x /content/archive_temp.zip -odata/processed/split_70_15_15_augmented -y > /dev/null
    !rm -f /content/archive_temp.zip
    if os.path.exists("data/processed/split_70_15_15_augmented/mvtec_augmented.zip"):
        !unzip -q data/processed/split_70_15_15_augmented/mvtec_augmented.zip -d data/processed/split_70_15_15_augmented/
        !rm -f data/processed/split_70_15_15_augmented/mvtec_augmented.zip

!ls -la data/processed/split_70_15_15_augmented

## Bước 4: Thực Nghiệm Huấn Luyện & Đánh Giá

### Cell 4A: Huấn Luyện BASELINE (YOLO11n Gốc)

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline \
    --name train

### Cell 4B: Huấn Luyện CBAM BACKBONE (Gộp Train + Đánh Giá Vào Gói `runs/backbone`)

In [ ]:
import shutil
from pathlib import Path

# 1. Huấn luyện CBAM Backbone
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone \
    --name train

# 2. Đồng bộ Baseline sang gói backbone
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 3. Xuất biểu đồ so sánh Baseline vs CBAM Backbone
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone

### Cell 4B_crack: Chạy Tiếp Tục CBAM BACKBONE Nếu Bị Crash Giữa Chừng
> *(Trước khi chạy: nếu bị ngắt phiên Colab, hãy chạy lại Bước 1 và Bước 3 trước)*

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

# Trỏ thẳng vào checkpoint last.pt trên Google Drive để tiếp tục
model = YOLO('/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone/train/weights/last.pt')
model.train(resume=True)

### Cell 4B_eval: Đánh Giá So Sánh CBAM BACKBONE (Chạy Sau Khi Vừa Resume Xong Cell 4B_crack)

In [ ]:
import shutil
from pathlib import Path

# 1. Đồng bộ Baseline sang gói backbone
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 2. Xuất biểu đồ so sánh Baseline vs CBAM Backbone
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone

### Cell 4C: Huấn Luyện CBAM NECK (Gộp Train + Đánh Giá Vào Gói `runs/neck`)
> Dành cho Thành viên 1: Chạy trọn gói từ Epoch 1. Nếu không bị crash, cell này sẽ tự động train xong và xuất biểu đồ vào `runs/neck/evaluation`.

In [ ]:
import shutil
from pathlib import Path

# 1. Huấn luyện CBAM Neck
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck \
    --name train

# 2. Đồng bộ Baseline sang gói neck
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 3. Xuất biểu đồ so sánh Baseline vs CBAM Neck vào runs/neck/evaluation
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck

### Cell 4C_crack: Chạy Tiếp Tục CBAM NECK Nếu Bị Crash Giữa Chừng
> *(Trước khi chạy: nếu bị ngắt phiên Colab, hãy chạy lại Bước 1 và Bước 3 trước)*

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

# Trỏ thẳng vào checkpoint last.pt trên Google Drive để tiếp tục
model = YOLO('/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck/train/weights/last.pt')
model.train(resume=True)

### Cell 4C_eval: Đánh Giá So Sánh CBAM NECK (Chạy Sau Khi Vừa Resume Xong Cell 4C_crack)
> Chỉ chạy cell này sau khi Cell 4C_crack đã hoàn thành đủ 100 epochs để xuất biểu đồ so sánh vào `runs/neck/evaluation`.

In [ ]:
import shutil
from pathlib import Path

# 1. Đồng bộ Baseline sang gói neck
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 2. Xuất biểu đồ so sánh Baseline vs CBAM Neck vào runs/neck/evaluation
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck

### Cell 4D: Huấn Luyện CBAM HEAD (Gộp Train + Đánh Giá Vào Gói `runs/head`)
> Dành cho Thành viên 2: Chạy trọn gói từ Epoch 1. Nếu không bị crash, cell này sẽ tự động train xong và xuất biểu đồ vào `runs/head/evaluation`.

In [ ]:
import shutil
from pathlib import Path

# 1. Huấn luyện CBAM Head
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head \
    --name train

# 2. Đồng bộ Baseline sang gói head
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 3. Xuất biểu đồ so sánh Baseline vs CBAM Head vào runs/head/evaluation
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head

### Cell 4D_crack: Chạy Tiếp Tục CBAM HEAD Nếu Bị Crash Giữa Chừng
> *(Trước khi chạy: nếu bị ngắt phiên Colab, hãy chạy lại Bước 1 và Bước 3 trước)*

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

# Trỏ thẳng vào checkpoint last.pt trên Google Drive để tiếp tục
model = YOLO('/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head/train/weights/last.pt')
model.train(resume=True)

### Cell 4D_eval: Đánh Giá So Sánh CBAM HEAD (Chạy Sau Khi Vừa Resume Xong Cell 4D_crack)
> Chỉ chạy cell này sau khi Cell 4D_crack đã hoàn thành đủ 100 epochs để xuất biểu đồ so sánh vào `runs/head/evaluation`.

In [ ]:
import shutil
from pathlib import Path

# 1. Đồng bộ Baseline sang gói head
BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

# 2. Xuất biểu đồ so sánh Baseline vs CBAM Head vào runs/head/evaluation
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head